# Tiny GPT — GPT-4 E1 Transformer Depth Experiments (Google Colab GPU Workflow)

This notebook provides the complete, automated GPU training, evaluation, and backup pipeline for **GPT-4 Experiment E1: Transformer Depth Scaling**.

### Scientific Objective
Evaluate the empirical scaling laws of Transformer depth ($L \in \{2, 3, 4, 6\}$) on next-token prediction under strictly controlled token budgets, architecture configurations, and optimization settings:
- **Baseline (GPT-3 Baseline):** 2 layers (60,800 parameters)
- **E1-A:** 3 layers (73,504 parameters, $\Delta = +12,704$)
- **E1-B:** 4 layers (86,208 parameters, $\Delta = +25,408$)
- **E1-C:** 6 layers (111,616 parameters, $\Delta = +50,816$)

### Fixed Experimental Controls Across All Depths:
- Vocabulary Size: `512` (subword BPE)
- Embedding Dimension: `32`
- Context Length: `64` tokens
- Attention Heads: `4` (head dim: 8)
- Feed-Forward Dimension: `128`
- Batch Size: `16` ($1,024$ tokens per optimizer step)
- Training Steps: `5,000` ($5,120,000$ total tokens)
- Learning Rate: $3.0 \times 10^{-4}$ (AdamW)
- Random Seed: `42` (with independent generators for training and validation)


---
## 0. Global Safety Controls & Execution Flags

> [!IMPORTANT]
> To prevent accidental launch of the full four-model training suite during a "Run All" operation, `RUN_FULL_EXPERIMENTS` is **strictly disabled by default**.
> Only the 20-step smoke test (`RUN_SMOKE_TEST = True`) will execute automatically.


In [ ]:
# ==============================================================================
# GLOBAL EXECUTION FLAGS
# ==============================================================================
RUN_SMOKE_TEST = True        # Executes isolated 20-step GPU smoke test in temporary storage
RUN_FULL_EXPERIMENTS = False  # SAFETY LOCK: Must be explicitly set to True by the user to train

print("=" * 65)
print(" EXECUTION CONTROLS CONFIGURED")
print("=" * 65)
print(f"  RUN_SMOKE_TEST:       {RUN_SMOKE_TEST}")
print(f"  RUN_FULL_EXPERIMENTS: {RUN_FULL_EXPERIMENTS}")
if not RUN_FULL_EXPERIMENTS:
    print("  STATUS:               >>> SAFETY LOCK ENGAGED <<<")
    print("                        Full 5,000-step training is disabled.")
print("=" * 65)


---
## Stage A — Environment Initialization

- Detect Google Colab execution environment.
- Verify repository directory and check out `feature/gpt4-architecture-experiments`.
- Verify Git branch and commit hash.
- Install only missing dependencies.
- Verify CUDA availability, GPU device name, VRAM, and PyTorch version.


In [ ]:
import os
import sys
import subprocess
import torch

IN_COLAB = 'google.colab' in sys.modules

print("=" * 65)
print(" STAGE A: ENVIRONMENT INITIALIZATION")
print("=" * 65)

# 1. Repository directory resolution
if IN_COLAB:
    print("[Colab] Google Colab environment detected.")
    repo_dir = "/content/tiny-gpt"
    if not os.path.exists(repo_dir):
        print("[Colab] Cloning repository from GitHub...")
        !git clone https://github.com/nandish029/tiny-gpt.git {repo_dir}
    %cd {repo_dir}
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)
else:
    print("[Local] Local workspace detected.")
    repo_dir = os.path.abspath(os.path.join(os.getcwd(), "..")) if "notebooks" in os.getcwd() else os.getcwd()
    if repo_dir not in sys.path:
        sys.path.insert(0, repo_dir)

print(f"Working Directory:    {os.getcwd()}")

# 2. Git branch and commit verification
try:
    current_branch = subprocess.check_output(["git", "branch", "--show-current"]).decode().strip()
    current_commit = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
    print(f"Git Branch:           {current_branch}")
    print(f"Current Commit:       {current_commit[:10]} ({current_commit})")
except Exception as e:
    print(f"Git info warning: {e}")

# 3. Dependency check (install only missing)
missing_deps = []
for pkg in ["yaml", "tokenizers"]:
    try:
        __import__(pkg)
    except ImportError:
        missing_deps.append(pkg)

if missing_deps:
    print(f"Installing missing dependencies: {missing_deps}...")
    !pip install -q pyyaml tokenizers
    print("Dependencies installed successfully.")
else:
    print("Dependencies:         All required packages are already installed.")

# 4. CUDA & GPU verification
cuda_available = torch.cuda.is_available()
print(f"PyTorch Version:      {torch.__version__}")
print(f"CUDA Available:       {cuda_available}")

if cuda_available:
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    device_count = torch.cuda.device_count()
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"GPU Accelerator:      {gpu_name} ({vram_gb:.2f} GB VRAM, {device_count} device(s))")
    print(f"CUDA Version:         {torch.version.cuda}")
    print(">>> GPU Acceleration is active and ready.")
else:
    device = torch.device("cpu")
    print("WARNING: CUDA is NOT available. Running on CPU.")
    if IN_COLAB:
        print(">>> Go to 'Runtime' > 'Change runtime type' and select 'T4 GPU'.")
print("=" * 65)


---
## Stage B — Repository Integrity Verification

- Confirm execution branch is `feature/gpt4-architecture-experiments`.
- Confirm working tree status.
- Verify existence of all four E1 YAML configurations.
- Verify processed dataset splits and tokenizer artifacts.
- Verify canonical GPT-1, GPT-2, and GPT-3 artifacts remain untouched.


In [ ]:
print("=" * 65)
print(" STAGE B: REPOSITORY INTEGRITY VERIFICATION")
print("=" * 65)

# 1. Branch verification
branch = subprocess.check_output(["git", "branch", "--show-current"]).decode().strip()
print(f"[Check 1/5] Active Branch: {branch}")
assert branch == "feature/gpt4-architecture-experiments", f"Unexpected branch: {branch}"

# 2. Working tree cleanliness check
git_status = subprocess.check_output(["git", "status", "--porcelain"]).decode().strip()
modified_tracked = [line for line in git_status.splitlines() if line.startswith(" M") or line.startswith("M ")]
print(f"[Check 2/5] Tracked modifications: {len(modified_tracked)} files")

# 3. Verify all four E1 YAML configurations exist
config_paths = {
    "baseline": "experiments/gpt4/e1_depth/configs/baseline.yaml",
    "e1_a":     "experiments/gpt4/e1_depth/configs/e1_a.yaml",
    "e1_b":     "experiments/gpt4/e1_depth/configs/e1_b.yaml",
    "e1_c":     "experiments/gpt4/e1_depth/configs/e1_c.yaml",
}
for name, path in config_paths.items():
    assert os.path.isfile(path), f"Missing E1 configuration: {path}"
print("[Check 3/5] All 4 E1 YAML configuration files verified.")

# 4. Verify dataset and tokenizer artifacts exist
data_files = [
    "data/processed/gpt3_tokenizer.json",
    "data/processed/train.jsonl",
    "data/processed/val.jsonl"
]
for df in data_files:
    assert os.path.isfile(df), f"Missing data file: {df}"
print("[Check 4/5] Processed TinyStories dataset splits and tokenizer verified.")

# 5. Verify canonical GPT-1/2/3 artifacts exist and are not touched
canonical_files = [
    "configs/gpt1.yaml",
    "configs/gpt2.yaml",
    "configs/gpt3.yaml",
    "checkpoints/gpt3/gpt3_baseline.pt"
]
for cf in canonical_files:
    assert os.path.isfile(cf), f"Missing canonical artifact: {cf}"
print("[Check 5/5] Canonical GPT-1, GPT-2, and GPT-3 artifacts verified untouched.")

print("=" * 65)
print("REPOSITORY INTEGRITY CONFIRMED")
print("=" * 65)


---
## Stage C — Experiment Preflight Inspection

- Load all four existing E1 configurations.
- Verify and display model dimensions and parameter counts.
- Prove that **only** `num_layers` differs across the four configurations.
- Prove that batch size, context length, learning rate, optimizer, seed, and token budgets are identical.
- Confirm all output paths are strictly isolated.


In [ ]:
from src.utils.config import load_config
from src.experiments.runner import ExperimentRunner

print("=" * 65)
print(" STAGE C: EXPERIMENT PREFLIGHT INSPECTION")
print("=" * 65)

loaded_configs = {name: load_config(p) for name, p in config_paths.items()}
baseline_cfg = loaded_configs["baseline"]

print(f"{'Config':<10} | {'Layers':<6} | {'Params':<10} | {'Batch':<6} | {'Context':<8} | {'LR':<8} | {'Seed':<5} | {'Steps':<6}")
print("-" * 75)

for name, cfg in loaded_configs.items():
    runner = ExperimentRunner(config_path=config_paths[name], device=str(device))
    insp = runner.inspect()
    m = cfg["model"]
    t = cfg["training"]
    
    # Assert shared architecture parameters match baseline exactly
    assert m["vocab_size"] == baseline_cfg["model"]["vocab_size"], "vocab_size mismatch"
    assert m["embedding_dim"] == baseline_cfg["model"]["embedding_dim"], "embedding_dim mismatch"
    assert m["max_context_length"] == baseline_cfg["model"]["max_context_length"], "context_length mismatch"
    assert m["num_heads"] == baseline_cfg["model"]["num_heads"], "num_heads mismatch"
    assert m["feed_forward_dim"] == baseline_cfg["model"]["feed_forward_dim"], "ffn_dim mismatch"
    
    # Assert shared training parameters match baseline exactly
    assert t["batch_size"] == baseline_cfg["training"]["batch_size"], "batch_size mismatch"
    assert float(t["learning_rate"]) == float(baseline_cfg["training"]["learning_rate"]), "learning_rate mismatch"
    assert t["train_steps"] == baseline_cfg["training"]["train_steps"], "train_steps mismatch"
    assert t["seed"] == baseline_cfg["training"]["seed"], "seed mismatch"
    
    # Assert output directories are disjoint
    if name != "baseline":
        assert insp["paths"]["checkpoint_dir"] != baseline_cfg["paths"]["checkpoint_dir"], f"Path collision in {name}"
    
    params = insp["parameters"]["trainable"]
    print(f"{name:<10} | {m['num_layers']:<6} | {params:<10,} | {t['batch_size']:<6} | {m['max_context_length']:<8} | {float(t['learning_rate']):<8.1e} | {t['seed']:<5} | {t['train_steps']:<6}")

print("-" * 75)
print("SCIENTIFIC CONTROLS VALIDATED: Only 'num_layers' varies across all 4 experiments.")
print(f"Token budget is identical for all experiments: 5,000 steps x 16 batch x 64 ctx = 5,120,000 tokens.")
print("=" * 65)


---
## Stage D — 20-Step GPU Smoke Test

- Gated by `RUN_SMOKE_TEST`.
- Runs exactly 20 optimizer steps on the 2-layer baseline configuration.
- Operates strictly inside an isolated temporary directory.
- Verifies finite training and validation loss values.
- Verifies gradient calculation and non-zero parameter updates.
- Saves and reloads a temporary checkpoint with strict state-dict matching.
- Measures average milliseconds per step and peak GPU memory allocation.


In [ ]:
import time
import math
import tempfile
from src.model.gpt import GPT

smoke_test_results = None

if RUN_SMOKE_TEST:
    print("=" * 65)
    print(" STAGE D: GPU SMOKE TEST (20 STEPS)")
    print("=" * 65)
    
    with tempfile.TemporaryDirectory(prefix="e1_smoke_gpu_") as tmpdir:
        temp_ckpt_dir = os.path.join(tmpdir, "checkpoints")
        temp_res_dir = os.path.join(tmpdir, "results")
        temp_ckpt_path = os.path.join(temp_ckpt_dir, "smoke_test.pt")
        
        runner = ExperimentRunner(config_path=config_paths["baseline"], device=str(device), seed=42)
        runner.checkpoint_dir = temp_ckpt_dir
        runner.results_dir = temp_res_dir
        runner.checkpoint_path = temp_ckpt_path
        runner.training_cfg["val_interval"] = 5  # validate every 5 steps
        
        # Build model and capture weight snapshot before step 1
        model = runner.build_model()
        initial_params = {name: p.clone().detach() for name, p in model.named_parameters()}
        
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
        
        smoke_steps = 20
        t_start = time.perf_counter()
        results = runner.train(steps=smoke_steps, overwrite=True)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t_end = time.perf_counter()
        
        elapsed_sec = t_end - t_start
        sec_per_step = elapsed_sec / smoke_steps
        ms_per_step = sec_per_step * 1000.0
        
        train_loss = results["final_train_loss"]
        val_loss = results["final_val_loss"]
        min_val = results["min_val_loss"]
        
        # 1. Assert finite losses
        assert math.isfinite(train_loss) and train_loss > 0, f"Invalid train loss: {train_loss}"
        assert math.isfinite(val_loss) and val_loss > 0, f"Invalid val loss: {val_loss}"
        
        # 2. Assert active gradients
        has_grad = any(p.grad is not None and torch.norm(p.grad).item() > 0 for p in model.parameters())
        assert has_grad, "Zero or missing gradients after backward pass!"
        
        # 3. Assert optimizer updates
        max_weight_delta = max(torch.norm(p.detach() - initial_params[name]).item() for name, p in model.named_parameters())
        assert max_weight_delta > 0.0, "Model parameters were not modified by optimizer!"
        
        # 4. Assert checkpoint saving and strict reload
        assert os.path.isfile(temp_ckpt_path), f"Checkpoint not saved to {temp_ckpt_path}"
        ckpt = torch.load(temp_ckpt_path, map_location=device, weights_only=False)
        assert ckpt["step"] == smoke_steps, f"Checkpoint step mismatch: {ckpt['step']}"
        
        reloaded_model = GPT(**runner.model_cfg).to(device)
        reloaded_model.load_state_dict(ckpt["model_state_dict"], strict=True)
        reloaded_model.eval()
        
        weights_identical = all(torch.equal(p, model.state_dict()[n]) for n, p in reloaded_model.named_parameters())
        assert weights_identical, "Reloaded weights diverged from trained in-memory weights!"
        
        # 5. Measure peak memory
        peak_vram_mb = 0.0
        if torch.cuda.is_available():
            peak_vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
        
        smoke_test_results = {
            "device": str(device),
            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
            "pytorch_version": torch.__version__,
            "smoke_steps": smoke_steps,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "min_val_loss": min_val,
            "max_weight_delta": max_weight_delta,
            "total_duration_sec": elapsed_sec,
            "sec_per_step": sec_per_step,
            "ms_per_step": ms_per_step,
            "peak_vram_mb": peak_vram_mb,
            "checkpoint_roundtrip": "PASSED (strict=True, bit-for-bit identical)"
        }
        
        print(f"Smoke Test:           {smoke_steps} steps completed successfully.")
        print(f"Final Train Loss:     {train_loss:.4f} (finite)")
        print(f"Final Val Loss:       {val_loss:.4f} (finite)")
        print(f"Max Weight Delta:     {max_weight_delta:.6f}")
        print(f"Step Duration:        {ms_per_step:.2f} ms/step ({sec_per_step:.4f} s/step)")
        if torch.cuda.is_available():
            print(f"Peak GPU VRAM:        {peak_vram_mb:.2f} MB")
        print(f"Checkpoint Roundtrip: {smoke_test_results['checkpoint_roundtrip']}")
        print("=" * 65)
else:
    print("Smoke test skipped (RUN_SMOKE_TEST = False).")


---
## Stage E — Results Summary & Runtime Projections

Displays a structured diagnostic report based on actual measured hardware performance and projects 5,000-step training durations across all four depth configurations.


In [ ]:
if smoke_test_results is not None:
    print("=" * 70)
    print(" STAGE E: SMOKE TEST SUMMARY & FULL RUNTIME PROJECTIONS")
    print("=" * 70)
    print(f"Execution Device:         {smoke_test_results['gpu_name']} ({smoke_test_results['device']})")
    print(f"PyTorch Version:          {smoke_test_results['pytorch_version']}")
    print(f"Optimizer Updates:        {smoke_test_results['smoke_steps']} steps")
    print(f"Train Loss (step 20):     {smoke_test_results['train_loss']:.4f}")
    print(f"Val Loss (step 20):       {smoke_test_results['val_loss']:.4f}")
    print(f"Peak GPU Memory:          {smoke_test_results['peak_vram_mb']:.2f} MB")
    print(f"Step Throughput:          {smoke_test_results['ms_per_step']:.2f} ms/step ({smoke_test_results['sec_per_step']:.4f} s/step)")
    print(f"Checkpoint Roundtrip:     {smoke_test_results['checkpoint_roundtrip']}")
    print("-" * 70)
    print("PROJECTED 5,000-STEP RUNTIMES (Based on measured hardware step time):")
    print(f"{'Experiment':<12} | {'Layers':<6} | {'Params':<10} | {'Layer Ratio':<12} | {'Est. Step (ms)':<14} | {'Est. 5k Steps':<15}")
    print("-" * 70)
    
    base_ms = smoke_test_results['ms_per_step']
    layer_counts = {"baseline": 2, "e1_a": 3, "e1_b": 4, "e1_c": 6}
    params_counts = {"baseline": 60800, "e1_a": 73504, "e1_b": 86208, "e1_c": 111616}
    
    total_projected_sec = 0.0
    for exp_name, l_count in layer_counts.items():
        # Compute scaling ratio: non-layer overhead + layer linear scaling
        scaling_factor = l_count / 2.0
        exp_ms = base_ms * (0.2 + 0.8 * scaling_factor)
        exp_5k_sec = (exp_ms / 1000.0) * 5000
        total_projected_sec += exp_5k_sec
        exp_min = exp_5k_sec / 60.0
        print(f"{exp_name:<12} | {l_count:<6} | {params_counts[exp_name]:<10,} | {scaling_factor:<12.1f} | {exp_ms:<14.2f} | {exp_min:.2f} min ({exp_5k_sec:.1f}s)")
        
    print("-" * 70)
    print(f"Total Projected Suite Runtime: {total_projected_sec/60.0:.2f} minutes ({total_projected_sec/3600.0:.2f} hours)")
    print("=" * 70)


---
## Stage F — Sequential Full-Training Controller (Future Execution)

> [!CAUTION]
> This controller executes the complete 4-model experimental suite (5,000 steps each).
> It requires `RUN_FULL_EXPERIMENTS = True` and will refuse to run by default.

### Key Controller Features:
- **Sequential Execution:** Trains `baseline` $\to$ `e1_a` $\to$ `e1_b` $\to$ `e1_c` in order.
- **Overwrite Protection:** Refuses to overwrite existing checkpoints without explicit parameter confirmation.
- **Atomic Verification:** Verifies checkpoint existence and size before advancing to the next experiment.
- **Automatic Drive Backup:** Immediately mirrors completed checkpoints and result JSONs to Google Drive.
- **Fail-Safe Halting:** If any experiment fails, execution stops immediately, preserving all previously completed experiment artifacts.


In [ ]:
import shutil

def run_full_e1_suite(
    run_experiments: bool = False,
    backup_to_drive: bool = True,
    drive_backup_dir: str = "/content/drive/MyDrive/tiny_gpt_e1_backups"
):
    """
    Sequential production controller for GPT-4 E1 depth scaling experiments.
    Executes baseline (2L), e1_a (3L), e1_b (4L), and e1_c (6L) sequentially.
    """
    if not run_experiments:
        raise RuntimeError("Safety Lock: run_experiments must be explicitly set to True.")
        
    experiments = ["baseline", "e1_a", "e1_b", "e1_c"]
    suite_start = time.time()
    completed_summaries = {}
    
    # Mount Google Drive if backup requested
    if backup_to_drive and IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        os.makedirs(drive_backup_dir, exist_ok=True)
        print(f"[Drive] Backup directory ready: {drive_backup_dir}")
        
    for exp_name in experiments:
        cfg_file = config_paths[exp_name]
        print(f"\n{'='*75}")
        print(f" STARTING EXPERIMENT: {exp_name.upper()} ({cfg_file})")
        print(f"{'='*75}")
        
        runner = ExperimentRunner(config_path=cfg_file, device=str(device))
        
        # Check overwrite protection
        if os.path.exists(runner.checkpoint_path):
            print(f"[Notice] Checkpoint already exists at {runner.checkpoint_path}.")
            print(f"[Notice] Skipping training for {exp_name} to preserve existing weights.")
            continue
            
        t0 = time.time()
        try:
            # 1. Train 5,000 steps
            train_results = runner.train(steps=5000, overwrite=False)
            t_train = time.time() - t0
            
            # 2. Verify checkpoint existence & non-empty size
            assert os.path.isfile(runner.checkpoint_path), f"Checkpoint missing for {exp_name}!"
            ckpt_size = os.path.getsize(runner.checkpoint_path)
            assert ckpt_size > 1000, f"Corrupted or empty checkpoint for {exp_name}!"
            print(f"[Checkpoint Verified] {runner.checkpoint_path} ({ckpt_size:,} bytes)")
            
            # 3. Formal post-training evaluation (50 batches on validation split)
            eval_metrics = runner.evaluate(batch_count=50, split="val", require_checkpoint=True)
            
            summary = {
                "experiment": exp_name,
                "layers": runner.model_cfg["num_layers"],
                "parameters": runner.count_parameters()["trainable"],
                "train_duration_sec": round(t_train, 2),
                "final_train_loss": train_results["final_train_loss"],
                "final_val_loss": train_results["final_val_loss"],
                "min_val_loss": train_results["min_val_loss"],
                "eval_loss_50_batches": eval_metrics["loss"],
                "eval_perplexity": eval_metrics["perplexity"],
                "checkpoint_path": runner.checkpoint_path,
                "checkpoint_size_bytes": ckpt_size
            }
            completed_summaries[exp_name] = summary
            
            print(f"\n[SUMMARY: {exp_name.upper()}]")
            print(f"  Training Time:   {t_train:.1f}s ({t_train/60:.2f} min)")
            print(f"  Eval Loss:       {eval_metrics['loss']:.4f}")
            print(f"  Eval Perplexity: {eval_metrics['perplexity']:.4f}")
            
            # 4. Immediate Drive Backup
            if backup_to_drive and IN_COLAB:
                exp_ckpt_dest = os.path.join(drive_backup_dir, "checkpoints", exp_name)
                exp_res_dest = os.path.join(drive_backup_dir, "results")
                os.makedirs(exp_ckpt_dest, exist_ok=True)
                os.makedirs(exp_res_dest, exist_ok=True)
                
                shutil.copy2(runner.checkpoint_path, exp_ckpt_dest)
                results_file = os.path.join(runner.results_dir, f"{exp_name}_results.json")
                if os.path.isfile(results_file):
                    shutil.copy2(results_file, exp_res_dest)
                    
                # Verify backup file exists
                backed_up_ckpt = os.path.join(exp_ckpt_dest, os.path.basename(runner.checkpoint_path))
                assert os.path.isfile(backed_up_ckpt), "Drive backup verification failed!"
                print(f"[Drive Backup] Verified at: {backed_up_ckpt}")
                
        except Exception as e:
            print(f"\n[ERROR] Experiment {exp_name} encountered an error: {e}")
            print("[HALTING] Sequence halted to protect completed artifacts.")
            raise e
            
    total_elapsed = time.time() - suite_start
    print(f"\n{'='*75}")
    print(f" ALL E1 EXPERIMENTS FINISHED (Total Time: {total_elapsed/60:.2f} min)")
    print(f"{'='*75}")
    return completed_summaries


### Execution Gate for Full Training


In [ ]:
# Full Training Safety Gate
if RUN_FULL_EXPERIMENTS:
    print("Full training flag is TRUE. Initiating sequential execution...")
    suite_results = run_full_e1_suite(run_experiments=True, backup_to_drive=True)
else:
    print("=" * 70)
    print("SAFETY LOCK ENGAGED: Full training was NOT launched.")
    print("  `RUN_FULL_EXPERIMENTS` is currently False.")
    print("  To train the 4 full models, explicitly set `RUN_FULL_EXPERIMENTS = True` in Cell 0.")
    print("=" * 70)


---
## Stage G — Colab Disconnection Protection & Manual Backup

Utility functions for manual synchronization or 1-click browser download of generated checkpoints and JSON summaries.


In [ ]:
# Option A: Manual synchronization to Google Drive
def manual_sync_to_drive(drive_dest="/content/drive/MyDrive/tiny_gpt_e1_backups"):
    if IN_COLAB:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        !cp -ru checkpoints/gpt4/e1_depth/ {drive_dest}/
        !cp -ru experiments/gpt4/e1_depth/results/ {drive_dest}/
        print(f"All checkpoints and results mirrored to Google Drive: {drive_dest}")
    else:
        print("Local environment: Artifacts are already stored locally on disk.")

# Option B: Create and trigger 1-click ZIP download
def download_artifacts_zip():
    archive_name = "tiny_gpt_e1_artifacts"
    shutil.make_archive(
        base_name=archive_name,
        format='zip',
        root_dir='.',
        base_dir='checkpoints/gpt4/e1_depth'
    )
    if IN_COLAB:
        from google.colab import files
        files.download(f"{archive_name}.zip")
        print("ZIP archive download initiated.")
    else:
        print(f"Archive created at {archive_name}.zip")
